# Figure 2: drought water-level decline and hydroclimatic drivers

This notebook regenerates the Fig. 2 draft outputs directly inside the notebook:

- Separate maps for 2012, 2022 and 2023 maximum WTD decline: drought-window deepest WTD minus pre-drought shallowest WTD.
- A monthly comparison of GRACE and reconstructed groundwater-storage proxy, plus raw-unit mean net infiltration, total pumping out, and mean precipitation.

Only PNG outputs are written to `outputs/figures/Fig2/`.


In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, ScalarFormatter
from pyproj import Transformer
from scipy.ndimage import generic_filter
from shapely.geometry import LineString


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'outputs' / 'RECON_MAIN_2011_2023').exists():
            return candidate
    raise FileNotFoundError('Could not find outputs/RECON_MAIN_2011_2023 from the current working directory.')


def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


ROOT = find_repo_root()
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
OUT_DIR = ROOT / 'outputs' / 'figures' / 'Fig2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RECON_MATRIX_PATH = RECON / 'reconstruction' / 'wtd_reconstructed_matrix.npy'
MONTH_INDEX_PATH = RECON / 'metadata' / 'month_index.csv'
GRID_LOOKUP_PATH = RECON / 'metadata' / 'grid_lookup.csv'
GRACE_RECON_MONTHLY_PATH = RECON / 'diagnostics' / 'regional_mass_correction' / 'regional_mass_correction_monthly.csv'
MRVA_BOUNDARY_PATH = ROOT / 'assets' / 'spatial' / 'mrva_boundary.geojson'
MISSISSIPPI_RIVER_GMT_PATH = ROOT / 'assets' / 'spatial' / 'mississippi_river.gmt'

SWB_MONTHLY_PATH = ROOT / 'data' / 'train_val_test_inputs' / 'GNN_spacetime' / 'H6' / 'swb_monthly.csv'
PUMPING_MONTHLY_PATH = ROOT / 'data' / 'train_val_test_inputs' / 'GNN_spacetime' / 'H6' / 'aiwum_monthly.csv'
PRECIP_DIR = ROOT / 'data' / '10 precipitation' / 'daymet_prcp_monthly_v4r1_mrva'
PRECIP_MATRIX_PATH = PRECIP_DIR / 'daymet_prcp_monthly_mrva_1km.npy'
PRECIP_MONTH_INDEX_PATH = PRECIP_DIR / 'month_index.csv'

DROUGHT_YEARS = (2012, 2022, 2023)
DECLINE_MAP_YEARS = (2012, 2017, 2022, 2023)
PRE_DROUGHT_MONTHS_BY_YEAR = {
    2012: (1, 4),
    2017: (1, 5),
    2022: (1, 5),
    2023: (1, 5),
}
DROUGHT_MONTHS_BY_YEAR = {
    2012: (5, 10),
    2017: (6, 11),
    2022: (6, 11),
    2023: (6, 11),
}
MAP_VMAX_PERCENTILE = 98.5
MAP_DISPLAY_MEDIAN_FILTER_SIZE = 3  # Display only; raw decline values are unchanged.
GRID_CRS = 'EPSG:5070'
LONLAT_CRS = 'EPSG:4326'
LONLAT_TICK_STEP_DEG = 1.0
MAP_BOUNDARY_COLOR = '#1f1f1f'
MAP_BOUNDARY_LW = 0.45
MAP_RIVER_COLOR = '#A9DCEF'
MAP_RIVER_LW = 0.45
NETINF_BAR_COLOR = '#9BD9CC'
PUMPING_BAR_COLOR = '#8A8176'
PRECIP_BAR_COLOR = '#CFE8C9'
NETINF_LABEL_COLOR = '#3B8F83'
PUMPING_LABEL_COLOR = '#5F584F'
PRECIP_LABEL_COLOR = '#5D8F58'
BAR_EDGE_COLOR = '#1A1A1A'
MONTH_BAR_WIDTH_DAYS = 23
PANEL_GRID_COLOR = '#D6D6D6'
PANEL_SPINE_COLOR = '#2A2A2A'
DROUGHT_SHADE_COLOR = '#C97C7C'
DROUGHT_SHADE_ALPHA = 0.055
EXPORT_DPI = 600

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.dpi': EXPORT_DPI,
    'savefig.bbox': 'tight',
    'axes.linewidth': 0.65,
    'xtick.major.width': 0.55,
    'ytick.major.width': 0.55,
    'xtick.major.size': 2.5,
    'ytick.major.size': 2.5,
})

print('Reconstruction:', display_path(RECON))
print('Output:', display_path(OUT_DIR))


In [ ]:
mat = np.load(RECON_MATRIX_PATH, mmap_mode='r')
month_index = pd.read_csv(MONTH_INDEX_PATH)
month_index['date'] = pd.to_datetime(month_index['month_label'])
grid_lookup = pd.read_csv(GRID_LOOKUP_PATH)


def grid_edge_lonlat() -> tuple[np.ndarray, np.ndarray]:
    x_by_col = grid_lookup.groupby('col')['x'].first().sort_index().to_numpy(dtype=np.float64)
    y_by_row = grid_lookup.groupby('row')['y'].first().sort_index().to_numpy(dtype=np.float64)
    dx = float(np.nanmedian(np.diff(x_by_col)))
    dy = float(np.nanmedian(np.diff(y_by_row)))
    x_edges = np.concatenate([[x_by_col[0] - 0.5 * dx], x_by_col + 0.5 * dx])
    y_edges = np.concatenate([[y_by_row[0] - 0.5 * dy], y_by_row + 0.5 * dy])
    xx, yy = np.meshgrid(x_edges, y_edges)
    transformer = Transformer.from_crs(GRID_CRS, LONLAT_CRS, always_xy=True)
    lon, lat = transformer.transform(xx, yy)
    return np.asarray(lon), np.asarray(lat)


def plot_line_geometry_lonlat(ax, geometry, **kwargs) -> None:
    if geometry.is_empty:
        return
    geom_type = geometry.geom_type
    if geom_type == 'LineString':
        x, y = geometry.xy
        ax.plot(np.asarray(x), np.asarray(y), **kwargs)
    elif geom_type in {'MultiLineString', 'GeometryCollection'}:
        for part in geometry.geoms:
            plot_line_geometry_lonlat(ax, part, **kwargs)
    elif geom_type == 'Polygon':
        plot_line_geometry_lonlat(ax, geometry.boundary, **kwargs)
    elif geom_type == 'MultiPolygon':
        for part in geometry.geoms:
            plot_line_geometry_lonlat(ax, part.boundary, **kwargs)


def load_mrva_boundary_polygon():
    boundary = gpd.read_file(MRVA_BOUNDARY_PATH).to_crs(LONLAT_CRS)
    if hasattr(boundary.geometry, 'union_all'):
        return boundary.geometry.union_all()
    return boundary.unary_union


def load_mrva_boundary_geometry():
    return load_mrva_boundary_polygon().boundary


def read_gmt_segments(path: Path) -> list[np.ndarray]:
    segments = []
    current = []
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if current:
                segments.append(np.asarray(current, dtype=np.float64))
                current = []
            continue
        lon, lat = line.split()[:2]
        current.append((float(lon), float(lat)))
    if current:
        segments.append(np.asarray(current, dtype=np.float64))
    return segments


def load_mississippi_segments_lonlat() -> list[np.ndarray]:
    return read_gmt_segments(MISSISSIPPI_RIVER_GMT_PATH)


def nice_degree_ticks(vmin: float, vmax: float, step: float = LONLAT_TICK_STEP_DEG) -> np.ndarray:
    start = np.ceil(vmin / step) * step
    stop = np.floor(vmax / step) * step
    return np.arange(start, stop + 0.5 * step, step)


def lon_label(value: float) -> str:
    hemi = 'W' if value < 0 else 'E'
    return f'{abs(value):.0f}\N{DEGREE SIGN}{hemi}'


def lat_label(value: float) -> str:
    hemi = 'S' if value < 0 else 'N'
    return f'{abs(value):.0f}\N{DEGREE SIGN}{hemi}'


MAP_LON_EDGES, MAP_LAT_EDGES = grid_edge_lonlat()
MAP_EXTENT_LONLAT = (
    float(np.nanmin(MAP_LON_EDGES)),
    float(np.nanmax(MAP_LON_EDGES)),
    float(np.nanmin(MAP_LAT_EDGES)),
    float(np.nanmax(MAP_LAT_EDGES)),
)
MRVA_POLYGON_GEOMETRY = load_mrva_boundary_polygon()
MRVA_BOUNDARY_GEOMETRY = MRVA_POLYGON_GEOMETRY.boundary
MISSISSIPPI_SEGMENTS_LONLAT = load_mississippi_segments_lonlat()


def clipped_segment_to_mrva(segment: np.ndarray):
    if len(segment) < 2:
        return None
    return LineString(segment).intersection(MRVA_POLYGON_GEOMETRY)


def month_window_indices(year: int, month_start: int, month_end: int) -> np.ndarray:
    mask = (
        (month_index['date'].dt.year == year)
        & month_index['date'].dt.month.between(month_start, month_end)
    )
    idx = np.flatnonzero(mask.to_numpy())
    if len(idx) == 0:
        raise ValueError(f'No months found for {year}-{month_start:02d} to {year}-{month_end:02d}.')
    return idx


def compute_peak_decline(year: int) -> np.ndarray:
    pre_idx = month_window_indices(year, *PRE_DROUGHT_MONTHS_BY_YEAR[year])
    drought_idx = month_window_indices(year, *DROUGHT_MONTHS_BY_YEAR[year])
    pre_shallow = np.nanmin(np.asarray(mat[pre_idx, :], dtype=np.float32), axis=0)
    drought_deep = np.nanmax(np.asarray(mat[drought_idx, :], dtype=np.float32), axis=0)
    decline = (drought_deep - pre_shallow).astype(np.float32)
    return decline


def rasterize_grid_values(values: np.ndarray) -> np.ndarray:
    n_rows = int(grid_lookup['row'].max()) + 1
    n_cols = int(grid_lookup['col'].max()) + 1
    out = np.full((n_rows, n_cols), np.nan, dtype=np.float32)
    out[grid_lookup['row'].to_numpy(dtype=int), grid_lookup['col'].to_numpy(dtype=int)] = values
    return out


def nanmedian_quiet(window: np.ndarray) -> float:
    finite = window[np.isfinite(window)]
    if finite.size == 0:
        return np.nan
    return float(np.median(finite))


def smooth_display_raster(raster: np.ndarray, filter_size: int = MAP_DISPLAY_MEDIAN_FILTER_SIZE) -> np.ndarray:
    if filter_size <= 1:
        return raster
    display = generic_filter(raster, nanmedian_quiet, size=filter_size, mode='nearest')
    display[~np.isfinite(raster)] = np.nan
    return display.astype(np.float32)


def build_decline_rasters() -> tuple[dict[int, np.ndarray], float]:
    decline_rasters = {}
    all_values = []
    for year in DECLINE_MAP_YEARS:
        decline = compute_peak_decline(year)
        decline_rasters[year] = rasterize_grid_values(decline)
        all_values.append(decline[np.isfinite(decline)])

    common_values = np.concatenate(all_values)
    vmax = float(np.nanpercentile(common_values, MAP_VMAX_PERCENTILE))
    return decline_rasters, max(vmax, 1.0)


def plot_decline_map(raster: np.ndarray, year: int, vmax: float, *, show_colorbar: bool = False) -> Path:
    cmap = LinearSegmentedColormap.from_list(
        'mrva_loss',
        ['#F7F7F7', '#F2B07D', '#B64342'],
        N=256,
    )
    cmap.set_bad('#FFFFFF')
    display_raster = smooth_display_raster(raster)

    fig_width = 3.35 if show_colorbar else 3.0
    fig, ax = plt.subplots(figsize=(fig_width, 5.8), dpi=EXPORT_DPI)
    image = ax.pcolormesh(
        MAP_LON_EDGES,
        MAP_LAT_EDGES,
        display_raster,
        cmap=cmap,
        vmin=0,
        vmax=vmax,
        shading='flat',
        rasterized=True,
    )
    for segment in MISSISSIPPI_SEGMENTS_LONLAT:
        clipped = clipped_segment_to_mrva(segment)
        if clipped is not None:
            plot_line_geometry_lonlat(ax, clipped, color=MAP_RIVER_COLOR, lw=MAP_RIVER_LW, alpha=0.95, zorder=3)
    plot_line_geometry_lonlat(ax, MRVA_BOUNDARY_GEOMETRY, color=MAP_BOUNDARY_COLOR, lw=MAP_BOUNDARY_LW, zorder=4)
    ax.set_xlim(MAP_EXTENT_LONLAT[0], MAP_EXTENT_LONLAT[1])
    ax.set_ylim(MAP_EXTENT_LONLAT[2], MAP_EXTENT_LONLAT[3])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(axis='both', which='both', bottom=False, left=False, labelbottom=False, labelleft=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    mean_lat = 0.5 * (MAP_EXTENT_LONLAT[2] + MAP_EXTENT_LONLAT[3])
    ax.set_aspect(1.0 / np.cos(np.deg2rad(mean_lat)))
    if show_colorbar:
        cbar = fig.colorbar(image, ax=ax, fraction=0.040, pad=0.025, extend='both')
        cbar.set_ticks([0, 1, 2, 3])
        cbar.set_ticklabels(['0', '1', '2', '3'])
        cbar.set_label('Maximum WTD decline (m)', rotation=90, labelpad=7)
        cbar.ax.tick_params(length=2.8, width=0.65, labelsize=10.5)
        cbar.outline.set_linewidth(0.6)
    out = OUT_DIR / f'Fig2_decline_{year}.png'
    fig.savefig(out, dpi=EXPORT_DPI)
    plt.close(fig)
    return out

print(f'Reconstruction matrix: {mat.shape[0]} months x {mat.shape[1]} grid cells')


In [ ]:
def compute_grace_reconstruction_series() -> pd.DataFrame:
    monthly = pd.read_csv(GRACE_RECON_MONTHLY_PATH)
    return pd.DataFrame({
        'date': pd.to_datetime(monthly['month_label']),
        'grace_lwe_anom_cm': monthly['grace_anom_cm'],
        'reconstruction_storage_proxy_anom_cm': monthly['corrected_storage_proxy_cm'],
        'has_grace_solution': monthly['has_valid_grace'].astype(bool),
    })


def compute_monthly_net_infiltration_mean() -> pd.DataFrame:
    swb = pd.read_csv(SWB_MONTHLY_PATH, usecols=['month_label', 'monthly_sum_net_infiltration'])
    out = (
        swb.groupby('month_label', as_index=False)['monthly_sum_net_infiltration']
        .mean()
        .sort_values('month_label')
    )
    out['date'] = pd.to_datetime(out['month_label'])
    return out.rename(columns={'monthly_sum_net_infiltration': 'net_infiltration_mean_mm'})[['date', 'net_infiltration_mean_mm']]


def compute_monthly_pumping_total() -> pd.DataFrame:
    pumping = pd.read_csv(PUMPING_MONTHLY_PATH, usecols=['month_label', 'monthly_pumping'])
    out = pumping.groupby('month_label', as_index=False)['monthly_pumping'].sum().sort_values('month_label')
    out['date'] = pd.to_datetime(out['month_label'])
    return out.rename(columns={'monthly_pumping': 'pumping_out_total_m3'})[['date', 'pumping_out_total_m3']]


def compute_monthly_precip_mean() -> pd.DataFrame:
    precip_month_index = pd.read_csv(PRECIP_MONTH_INDEX_PATH)
    precip_month_index['date'] = pd.to_datetime(precip_month_index['month_label'])
    precip = np.load(PRECIP_MATRIX_PATH, mmap_mode='r')
    precip_mean = np.nanmean(np.asarray(precip, dtype=np.float32), axis=1)
    out = precip_month_index[['date']].copy()
    out['precipitation_mean_mm'] = precip_mean
    return out


def build_driver_timeseries() -> pd.DataFrame:
    series = compute_grace_reconstruction_series()
    series = series.merge(compute_monthly_net_infiltration_mean(), on='date', how='left')
    series = series.merge(compute_monthly_pumping_total(), on='date', how='left')
    series = series.merge(compute_monthly_precip_mean(), on='date', how='left')
    return series


def annotate_drought_years(ax: plt.Axes) -> None:
    for year in DROUGHT_YEARS:
        start_month, end_month = DROUGHT_MONTHS_BY_YEAR[year]
        start_date = pd.Timestamp(year, start_month, 1)
        end_date = pd.Timestamp(year, end_month, 1) + pd.offsets.MonthEnd(0)
        ax.axvspan(start_date, end_date, color=DROUGHT_SHADE_COLOR, alpha=DROUGHT_SHADE_ALPHA, lw=0)


def style_driver_axis(ax: plt.Axes, *, show_bottom_labels: bool = False) -> None:
    ax.set_axisbelow(True)
    ax.grid(axis='both', color=PANEL_GRID_COLOR, lw=0.45, alpha=0.62, zorder=0)
    for side in ['top', 'right', 'bottom', 'left']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color(PANEL_SPINE_COLOR)
        ax.spines[side].set_linewidth(0.72)
    ax.tick_params(
        axis='x',
        which='major',
        top=True,
        bottom=True,
        labeltop=False,
        labelbottom=show_bottom_labels,
        direction='out',
        length=3.0,
        width=0.65,
        pad=2.0,
    )
    ax.tick_params(
        axis='y',
        which='major',
        right=True,
        labelright=False,
        direction='out',
        length=3.0,
        width=0.65,
        pad=2.0,
    )


def add_panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(-0.125, 1.02, label, transform=ax.transAxes, ha='right', va='bottom', fontsize=10, fontweight='bold')


def plot_driver_timeseries(series: pd.DataFrame, *, reconstruction_style: str = 'line', output_name: str = 'Fig2_GRACE_reconstruction_drivers.png') -> Path:
    valid = series['has_grace_solution'] & series['grace_lwe_anom_cm'].notna()
    colors = {
        'grace': '#6FA9B8',
        'grace_fill': '#EEF8F6',
        'recon': '#7A5538',
    }

    fig, axes = plt.subplots(
        4,
        1,
        figsize=(7, 7),
        dpi=EXPORT_DPI,
        sharex=True,
        gridspec_kw={'hspace': 0.14, 'height_ratios': [1.0, 0.82, 0.82, 1.0]},
    )
    fig.subplots_adjust(left=0.165, right=0.915, top=0.965, bottom=0.125)

    for ax in axes:
        style_driver_axis(ax, show_bottom_labels=(ax is axes[-1]))

    precip_ax = axes[0]
    precip_ax.bar(
        series['date'],
        series['precipitation_mean_mm'],
        width=MONTH_BAR_WIDTH_DAYS,
        color=PRECIP_BAR_COLOR,
        edgecolor='none',
        linewidth=0.0,
        align='center',
        zorder=3,
    )
    precip_ax.axhline(0, color=BAR_EDGE_COLOR, lw=0.65)
    precip_ax.set_ylabel('Precipitation\n(mm)')
    precip_ax.yaxis.label.set_color(PRECIP_LABEL_COLOR)

    ax = axes[3]
    recon_ax = ax.twinx()
    recon_values = series['reconstruction_storage_proxy_anom_cm'].to_numpy(dtype=float)
    if reconstruction_style == 'filled_area':
        dates = series['date'].to_numpy()
        # Diverging blue-white-red palette: zero is near white, as in the reference figure.
        signed_cmap = mpl.colormaps['RdBu']
        for idx in range(len(recon_values) - 1):
            y0, y1 = recon_values[idx], recon_values[idx + 1]
            if not np.isfinite(y0) or not np.isfinite(y1):
                continue
            scale = min(max(max(abs(y0), abs(y1)) / 20.0, 0.0), 1.0)
            if max(y0, y1) > 0.0:
                level = 0.50 + 0.46 * scale
                recon_ax.fill_between(dates[idx:idx + 2], 0.0, np.maximum([y0, y1], 0.0), color=signed_cmap(level), alpha=0.82, linewidth=0.0, zorder=2)
            if min(y0, y1) < 0.0:
                level = 0.50 - 0.46 * scale
                recon_ax.fill_between(dates[idx:idx + 2], 0.0, np.minimum([y0, y1], 0.0), color=signed_cmap(level), alpha=0.88, linewidth=0.0, zorder=2)
        recon_line, = recon_ax.plot(series['date'], recon_values, color=colors['recon'], lw=0.75, label='Reconstruction', zorder=3)
    elif reconstruction_style == 'line':
        recon_line, = recon_ax.plot(series['date'], recon_values, color=colors['recon'], lw=1.5, label='Reconstruction', zorder=3)
    else:
        raise ValueError(f'Unknown reconstruction_style: {reconstruction_style}')
    grace_points = ax.scatter(
        series.loc[valid, 'date'],
        series.loc[valid, 'grace_lwe_anom_cm'],
        s=10,
        facecolor=colors['grace_fill'],
        edgecolor=colors['grace'],
        linewidth=1.5,
        label='GRACE',
        zorder=3,
    )
    ax.axhline(0, color='#666666', lw=0.45)
    ax.set_ylabel('GRACE\nanomaly (cm)')
    ax.set_ylim(-40, 40)
    ax.tick_params(axis='y', colors=colors['grace'], right=False)
    ax.yaxis.label.set_color(colors['grace'])
    ax.spines['right'].set_visible(False)
    recon_ax.set_ylabel('Reconstruction\nanomaly (cm)')
    recon_ax.set_ylim(-20, 20)
    recon_ax.grid(False)
    recon_ax.tick_params(axis='x', top=False, bottom=False, labelbottom=False)
    recon_ax.tick_params(axis='y', colors=colors['recon'], direction='out', length=3.0, width=0.65, pad=2.0)
    recon_ax.yaxis.label.set_color(colors['recon'])
    for side in ['top', 'bottom', 'left']:
        recon_ax.spines[side].set_visible(False)
    recon_ax.spines['right'].set_visible(True)
    recon_ax.spines['right'].set_color(PANEL_SPINE_COLOR)
    recon_ax.spines['right'].set_linewidth(0.72)
    legend = ax.legend(
        [recon_line, grace_points],
        ['Reconstruction', 'GRACE'],
        frameon=True,
        ncol=2,
        loc='upper left',
        handlelength=1.55,
        columnspacing=0.95,
        borderpad=0.25,
        labelspacing=0.25,
        fontsize=7.5,
    )
    legend.get_frame().set_edgecolor('#B7B7B7')
    legend.get_frame().set_linewidth(0.55)
    legend.get_frame().set_facecolor('white')
    legend.get_frame().set_alpha(0.92)
    axes[1].bar(
        series['date'],
        series['net_infiltration_mean_mm'],
        width=MONTH_BAR_WIDTH_DAYS,
        color=NETINF_BAR_COLOR,
        edgecolor='none',
        linewidth=0.0,
        align='center',
        zorder=3,
    )
    axes[1].set_ylabel('Net infiltration\n(mm)')
    axes[1].yaxis.label.set_color(NETINF_LABEL_COLOR)
    axes[1].axhline(0, color='#666666', lw=0.45)

    axes[2].bar(
        series['date'],
        series['pumping_out_total_m3'],
        width=MONTH_BAR_WIDTH_DAYS,
        color=PUMPING_BAR_COLOR,
        edgecolor='none',
        linewidth=0.0,
        align='center',
        zorder=3,
    )
    axes[2].set_ylabel('Pumping\n(10$^9$ m$^3$)')
    axes[2].yaxis.label.set_color(PUMPING_LABEL_COLOR)
    axes[2].yaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value / 1e9:g}'))
    axes[2].axhline(0, color='#666666', lw=0.45)

    axes[-1].xaxis.set_major_locator(mdates.YearLocator(1))
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[-1].set_xlim(pd.Timestamp('2011-01-01'), pd.Timestamp('2023-12-31'))
    for tick in axes[-1].get_xticklabels():
        tick.set_rotation(45)
        tick.set_ha('right')

    out = OUT_DIR / output_name
    fig.savefig(out, dpi=EXPORT_DPI)
    plt.close(fig)
    return out


In [ ]:
decline_rasters, common_vmax = build_decline_rasters()
fig_paths = [
    plot_decline_map(
        decline_rasters[year],
        year,
        common_vmax,
        show_colorbar=(year == DECLINE_MAP_YEARS[-1]),
    )
    for year in DECLINE_MAP_YEARS
]
driver_series = build_driver_timeseries()
fig_paths.append(plot_driver_timeseries(driver_series))
fig_paths.append(plot_driver_timeseries(driver_series, reconstruction_style='filled_area', output_name='Fig2_GRACE_reconstruction_drivers_filled_trial.png'))

print('Fig2 outputs written to:')
for path in fig_paths:
    print(' ', display_path(path))


In [ ]:
try:
    from IPython.display import Image, display
except ImportError:
    Image = None
    display = None

FIG_TO_DISPLAY = 6
fig_to_show = fig_paths[FIG_TO_DISPLAY - 1]

print(display_path(fig_to_show))
if Image is not None:
    display(Image(filename=str(fig_to_show)))


## Source numbers for the Results text

This cell documents where the numerical values used in the Fig. 2 Results text come from. Decline statistics use the raw annual decline values, not the display-only median-filtered rasters. Positive decline means that reconstructed depth to water became deeper during the drawdown window.

In [ ]:
SOURCE_NUMBERS_CSV = OUT_DIR / 'Fig2_source_numbers.csv'


def append_source_number(rows, group, metric, value, *, unit='', year=None, source_note=''):
    rows.append({
        'group': group,
        'year': year,
        'metric': metric,
        'value': float(value) if pd.notna(value) else np.nan,
        'unit': unit,
        'source_note': source_note,
    })


rows = []

# A-D: annual maximum WTD decline maps.
for year in DECLINE_MAP_YEARS:
    pre_window = PRE_DROUGHT_MONTHS_BY_YEAR[year]
    drawdown_window = DROUGHT_MONTHS_BY_YEAR[year]
    decline = compute_peak_decline(year)
    finite = decline[np.isfinite(decline)]
    note = (
        f'Raw, unsmoothed grid-cell values. Decline = max WTD in {year}-{drawdown_window[0]:02d}'
        f'..{year}-{drawdown_window[1]:02d} minus min WTD in {year}-{pre_window[0]:02d}'
        f'..{year}-{pre_window[1]:02d}.'
    )
    append_source_number(rows, 'decline_map', 'pre_window_start_month', pre_window[0], year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'pre_window_end_month', pre_window[1], year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'drawdown_window_start_month', drawdown_window[0], year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'drawdown_window_end_month', drawdown_window[1], year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'n_grid_cells', decline.size, year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'n_valid_grid_cells', finite.size, year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'mean_decline_m', np.nanmean(finite), unit='m', year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'median_decline_m', np.nanmedian(finite), unit='m', year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'p25_decline_m', np.nanpercentile(finite, 25), unit='m', year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'p75_decline_m', np.nanpercentile(finite, 75), unit='m', year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'p90_decline_m', np.nanpercentile(finite, 90), unit='m', year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'p95_decline_m', np.nanpercentile(finite, 95), unit='m', year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'p98_5_decline_m', np.nanpercentile(finite, 98.5), unit='m', year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'max_decline_m', np.nanmax(finite), unit='m', year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'area_decline_gt_1m_pct', 100 * np.mean(finite > 1.0), unit='%', year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'area_decline_gt_2m_pct', 100 * np.mean(finite > 2.0), unit='%', year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'area_decline_gt_3m_pct', 100 * np.mean(finite > 3.0), unit='%', year=year, source_note=note)
    append_source_number(rows, 'decline_map', 'area_decline_le_0m_pct', 100 * np.mean(finite <= 0.0), unit='%', year=year, source_note=note)

# E: GRACE versus reconstructed regional anomaly.
driver_series_for_stats = driver_series if 'driver_series' in globals() else build_driver_timeseries()
valid_grace = (
    driver_series_for_stats['has_grace_solution']
    & driver_series_for_stats['grace_lwe_anom_cm'].notna()
    & driver_series_for_stats['reconstruction_storage_proxy_anom_cm'].notna()
)
grace_recon = driver_series_for_stats.loc[
    valid_grace,
    ['grace_lwe_anom_cm', 'reconstruction_storage_proxy_anom_cm'],
]
grace_note = 'Computed over months with valid GRACE/GRACE-FO solutions only.'
append_source_number(rows, 'grace_reconstruction', 'n_valid_months', len(grace_recon), source_note=grace_note)
append_source_number(rows, 'grace_reconstruction', 'pearson_r', grace_recon.corr(method='pearson').iloc[0, 1], source_note=grace_note)
append_source_number(rows, 'grace_reconstruction', 'spearman_rho', grace_recon.corr(method='spearman').iloc[0, 1], source_note=grace_note)
append_source_number(rows, 'grace_reconstruction', 'grace_std_cm', grace_recon['grace_lwe_anom_cm'].std(), unit='cm', source_note=grace_note)
append_source_number(rows, 'grace_reconstruction', 'reconstruction_std_cm', grace_recon['reconstruction_storage_proxy_anom_cm'].std(), unit='cm', source_note=grace_note)
append_source_number(rows, 'grace_reconstruction', 'grace_min_cm', grace_recon['grace_lwe_anom_cm'].min(), unit='cm', source_note=grace_note)
append_source_number(rows, 'grace_reconstruction', 'grace_max_cm', grace_recon['grace_lwe_anom_cm'].max(), unit='cm', source_note=grace_note)
append_source_number(rows, 'grace_reconstruction', 'reconstruction_min_cm', grace_recon['reconstruction_storage_proxy_anom_cm'].min(), unit='cm', source_note=grace_note)
append_source_number(rows, 'grace_reconstruction', 'reconstruction_max_cm', grace_recon['reconstruction_storage_proxy_anom_cm'].max(), unit='cm', source_note=grace_note)

# E: annual and drawdown-window hydroclimatic / pumping summaries.
driver_stats = driver_series_for_stats.copy()
driver_stats['year'] = driver_stats['date'].dt.year
driver_stats['month'] = driver_stats['date'].dt.month
for year in DECLINE_MAP_YEARS:
    drawdown_window = DROUGHT_MONTHS_BY_YEAR[year]
    annual = driver_stats.loc[driver_stats['year'] == year]
    drawdown = annual.loc[annual['month'].between(*drawdown_window)]
    note = 'Driver values are from the monthly series plotted in Fig. 2E.'
    append_source_number(rows, 'drivers', 'annual_precipitation_total_spatial_mean_mm', annual['precipitation_mean_mm'].sum(), unit='mm', year=year, source_note=note)
    append_source_number(rows, 'drivers', 'annual_net_infiltration_total_spatial_mean_mm', annual['net_infiltration_mean_mm'].sum(), unit='mm', year=year, source_note=note)
    append_source_number(rows, 'drivers', 'annual_pumping_total_1e9_m3', annual['pumping_out_total_m3'].sum() / 1e9, unit='10^9 m3', year=year, source_note=note)
    append_source_number(rows, 'drivers', 'drawdown_precipitation_total_spatial_mean_mm', drawdown['precipitation_mean_mm'].sum(), unit='mm', year=year, source_note=note)
    append_source_number(rows, 'drivers', 'drawdown_net_infiltration_total_spatial_mean_mm', drawdown['net_infiltration_mean_mm'].sum(), unit='mm', year=year, source_note=note)
    append_source_number(rows, 'drivers', 'drawdown_pumping_total_1e9_m3', drawdown['pumping_out_total_m3'].sum() / 1e9, unit='10^9 m3', year=year, source_note=note)

source_numbers = pd.DataFrame(rows)
source_numbers.to_csv(SOURCE_NUMBERS_CSV, index=False)

print(f'Wrote source-number table: {display_path(SOURCE_NUMBERS_CSV)}')
display(source_numbers.head(12))
